In [ ]:
import os
os.environ["PYSPARK_ALLOW_INSECURE_GATEWAY"] = "1"


from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, length, upper
from pyspark.sql.types import StructType, StringType, IntegerType


DB_URL = "jdbc:postgresql://postgres:5432/postgres"
DB_USER = "myuser"
DB_PASS = "myuserpass"
DB_TABLE = "kafka_data_02"
BAD_ROWS_PATH = "/tmp/bad_rows_02"
CHECKPOINT_PATH = "/tmp/checkpoints/kafka-to-pgsql_02"


# SparkSession z obsługą Kafka + JSON
spark = (
    SparkSession.builder
    .appName("StreamingJSONTransform")
    .master("local[*]")
    .getOrCreate()
)

# Schemat danych JSON
json_schema = (
    StructType()
    .add("id", IntegerType())
    .add("name", StringType())
    .add("message", StringType())
)

# Strumień z Kafka
df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "spark-lab2-topic")
    .option("startingOffsets", "latest")
    .option("badRecordsPath", BAD_ROWS_PATH)
    .load()
)

# Parsowanie JSON z kolumny value
parsed = (
    df.selectExpr("CAST(value AS STRING) as json_str")
    .select(from_json(col("json_str"), json_schema).alias("data"))
    .select("data.*")
)

# Przekształcanie danych
transformed = (
    parsed
    .withColumn("name_upper", upper(col("name")))
    .withColumn("msg_length", length(col("message")))
    .filter(col("msg_length") > 5)
)

# Zapisywanie do PostgreSQL
def write_to_postgres(batch_df, batch_id):
    (
        batch_df.write
        .format("jdbc")
        .option("url", DB_URL)
        .option("dbtable", DB_TABLE)
        .option("user", DB_USER)
        .option("password", DB_PASS)
        .option("driver", "org.postgresql.Driver")
        .mode("append")
        .save()
    )

# Zapis jako foreachBatch
query = (
    transformed.writeStream
    .foreachBatch(write_to_postgres)
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .start()
)

query.awaitTermination()
